# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [11]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [12]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['GZULNMDDVM', 'QWEXJUZGCO'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 7, 26, 21, 12, 14, 13,  4,  4, 22, 13],
       [17, 23,  5, 24, 10, 21, 26,  7,  3, 15]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 13, 22,  4,  4, 13, 14, 12, 21, 26],
       [ 0, 15,  3,  7, 26, 21, 10, 24,  5, 23]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[13, 22,  4,  4, 13, 14, 12, 21, 26,  7],
       [15,  3,  7, 26, 21, 10, 24,  5, 23, 17]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [13]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        

    def call(self, enc_ids, dec_ids):
        # 1. 编码器：词嵌入 + RNN前向传播
        enc_emb = self.embed_layer(enc_ids)              # (b_sz, enc_len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)       # enc_out: (b_sz, enc_len, h_sz)
        
        # 2. 解码器：词嵌入 + RNN前向传播，用编码器最终状态初始化
        dec_emb = self.embed_layer(dec_ids)              # (b_sz, dec_len, emb_sz)
        dec_out, dec_state = self.decoder(dec_emb, initial_state=enc_state)
        # dec_out: (b_sz, dec_len, h_sz)
        
        # 3. 双线性注意力：将编码器输出通过线性变换后与解码器输出计算注意力分数
        # dense_attn将enc_out映射: (b_sz, enc_len, h_sz)
        enc_mapped = self.dense_attn(enc_out)            # (b_sz, enc_len, h_sz)
        
        # 计算注意力分数: dec_out @ enc_mapped^T -> (b_sz, dec_len, enc_len)
        attn_scores = tf.matmul(dec_out, enc_mapped, transpose_b=True)
        
        # softmax归一化得到注意力权重
        attn_weights = tf.nn.softmax(attn_scores, axis=-1)  # (b_sz, dec_len, enc_len)
        
        # 4. 用注意力权重对编码器输出加权求和，得到上下文向量
        context = tf.matmul(attn_weights, enc_out)       # (b_sz, dec_len, h_sz)
        
        # 5. 将上下文向量与解码器输出拼接（相加）后映射到词表
        dec_combined = dec_out + context                  # (b_sz, dec_len, h_sz)
        logits = self.dense(dec_combined)                 # (b_sz, dec_len, v_sz)
        return logits
    
    

    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,] 
        单步解码：每次生成一个token
        '''
        # 1. 对当前token进行词嵌入
        inp_emb = self.embed_layer(x)                    # (b_sz, emb_sz)
        
        # 2. 解码器RNN单步计算，得到当前隐状态
        h, state = self.decoder_cell.call(inp_emb, state) # h: (b_sz, h_sz)
        
        # 3. 计算双线性注意力
        enc_mapped = self.dense_attn(enc_out)             # (b_sz, enc_len, h_sz)
        # h需要扩展维度后与enc_mapped做矩阵乘法
        attn_scores = tf.matmul(enc_mapped, tf.expand_dims(h, axis=-1))  # (b_sz, enc_len, 1)
        attn_scores = tf.squeeze(attn_scores, axis=-1)    # (b_sz, enc_len)
        attn_weights = tf.nn.softmax(attn_scores, axis=-1) # (b_sz, enc_len)
        
        # 4. 加权求和得到上下文向量
        context = tf.matmul(tf.expand_dims(attn_weights, axis=1), enc_out) # (b_sz, 1, h_sz)
        context = tf.squeeze(context, axis=1)              # (b_sz, h_sz)
        
        # 5. 将上下文向量与解码器隐状态相加，映射到词表
        dec_combined = h + context                         # (b_sz, h_sz)
        logits = self.dense(dec_combined)                  # (b_sz, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, state

# Loss函数以及训练逻辑

In [14]:

def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses


def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [15]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.2973099
step 500 : loss 1.4482353
step 1000 : loss 0.79734313
step 1500 : loss 0.35644755


<tf.Tensor: shape=(), dtype=float32, numpy=0.1802697479724884>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [16]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, False, False, True, False, False, False, False, False, False, True, True, False, False, False, False, True, False, True, True, False, True, False, True, True, True, True, False]
[('ECTPYPQRMPXYWNGTHBUT', 'TUBHTGNWYXPMRQPYPTCE'), ('TMZEGQIKZIVNUXXYFLQT', 'TLSJLQXNNZIPQIQOEAMB'), ('VTAUTXRCETKYMXDENUJD', 'NTUNEUNMYKFECRXSIAPV'), ('FVUGJROFYBDPRGMJJBBG', 'BMBTJEGRPDBYFORJGUVF'), ('IADCNNZOOBJRWGSZHRVE', 'EVRHZSGWRJBOOZNNCDAI'), ('KZTRJCEYAHIKWZOMZGRU', 'MRGXIGZRKIHAYEPJCTZK'), ('DVUCRXPMVTZIJMXPDHYH', 'HYHDPXMJIZTVMPXRCUVD'), ('TKWDGGZNGGYXUESXORCK', 'ZVBOWPHIAYGGDZFFNWKT'), ('UUINOIOPXKEMQDGSKJZA', 'QGINLXIROFKUPOEONNUX'), ('JGJYFTXEESFARVFGOQYI', 'IYQOOFVROFSEEXTFSJGJ'), ('KSDSEEIGNJYWSMSZRRWL', 'RRLWPSESDSJHGIVDPNSK'), ('FKMHIAZXQNFUQWHYYVOJ', 'EPVYYQHQUFNQXZAOHMKF'), ('PTQHMDJKQCZVTMWJJWET', 'TEWJJWMTVZCQKJDMHQTP'), ('PSYUEOVGPPUKXYDHDKDU', 'AYRKODVWKUPPGVMGZLSP'), ('JWWECVXCSEAIWWFCIDXH', 'HXDICFWWIAESCXVCEWWJ'), ('EYDYHSYMZNATDNAEDUBB', 'BGUDGAHDTHNZMY